# Évaluation TTS pour SVI téléphonique

Comparaison **Kokoro TTS** vs **Piper TTS** sur des phrases types de SVI en français.

- Kokoro : meilleure qualité, nécessite GPU
- Piper : plus rapide, tourne sur CPU

Les deux sont évalués en qualité originale (16kHz) ET en qualité téléphone (G.711 8kHz).

> **Runtime** : sélectionner GPU dans Runtime → Change runtime type → T4 GPU

## 1. Installation

In [ ]:
!pip install -q kokoro soundfile piper-tts pathvalidate
!apt-get install -qq ffmpeg > /dev/null 2>&1
print("Installation termin\u00e9e.")

## 2. Chargement des phrases

In [ ]:
import os
from pathlib import Path

# Télécharger phrases.txt depuis le repo
!wget -q https://raw.githubusercontent.com/chris-lmd/eval-tts-svi/master/phrases.txt -O phrases.txt

def charger_phrases(prefixe: str) -> list[tuple[str, str]]:
    phrases = []
    for line in Path("phrases.txt").read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        id_phrase, texte = line.split("|", 1)
        id_phrase = id_phrase.strip()
        texte = texte.strip()
        if id_phrase.startswith(prefixe):
            phrases.append((id_phrase, texte))
    return phrases

phrases_bot = charger_phrases("bot_")
print(f"{len(phrases_bot)} phrases chargées :")
for id_p, texte in phrases_bot:
    print(f"  {id_p}: {texte}")

## 3. G\u00e9n\u00e9ration Kokoro TTS (GPU)

In [ ]:
import time
import soundfile as sf
from kokoro import KPipeline

# Initialiser Kokoro en fran\u00e7ais
kokoro_pipeline = KPipeline(lang_code="f")

# Lister les voix fran\u00e7aises disponibles
print("Pipeline Kokoro initialis\u00e9.")
print("Voix par d\u00e9faut utilis\u00e9e (premi\u00e8re voix fran\u00e7aise disponible).")

In [ ]:
os.makedirs("samples/kokoro", exist_ok=True)

kokoro_times = {}

for id_phrase, texte in phrases_bot:
    output_file = f"samples/kokoro/{id_phrase}.wav"
    start = time.perf_counter()

    generator = kokoro_pipeline(texte, voice="ff_siwis")
    # Concaténer tous les segments
    import numpy as np
    segments = []
    for _, _, audio in generator:
        segments.append(audio)
    full_audio = np.concatenate(segments) if len(segments) > 1 else segments[0]
    sf.write(output_file, full_audio, 24000)

    elapsed_ms = (time.perf_counter() - start) * 1000
    kokoro_times[id_phrase] = elapsed_ms
    print(f"  {id_phrase}: {elapsed_ms:.0f}ms")

print(f"\nMoyenne: {sum(kokoro_times.values())/len(kokoro_times):.0f}ms")

## 4. G\u00e9n\u00e9ration Piper TTS (CPU)

In [ ]:
import subprocess

# T\u00e9l\u00e9charger le mod\u00e8le Piper fran\u00e7ais
os.makedirs("models", exist_ok=True)
if not os.path.exists("models/fr_FR-siwis-medium.onnx"):
    !wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/siwis/medium/fr_FR-siwis-medium.onnx -O models/fr_FR-siwis-medium.onnx
    !wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/siwis/medium/fr_FR-siwis-medium.onnx.json -O models/fr_FR-siwis-medium.onnx.json
    print("Mod\u00e8le Piper t\u00e9l\u00e9charg\u00e9.")
else:
    print("Mod\u00e8le Piper d\u00e9j\u00e0 pr\u00e9sent.")

In [ ]:
os.makedirs("samples/piper", exist_ok=True)

piper_times = {}

for id_phrase, texte in phrases_bot:
    output_file = f"samples/piper/{id_phrase}.wav"
    start = time.perf_counter()

    subprocess.run(
        ["piper", "--model", "models/fr_FR-siwis-medium.onnx",
         "--output_file", output_file],
        input=texte, text=True, check=True,
        capture_output=True,
    )

    elapsed_ms = (time.perf_counter() - start) * 1000
    piper_times[id_phrase] = elapsed_ms
    print(f"  {id_phrase}: {elapsed_ms:.0f}ms")

print(f"\nMoyenne: {sum(piper_times.values())/len(piper_times):.0f}ms")

## 5. Conversion qualit\u00e9 t\u00e9l\u00e9phone (G.711 8kHz)

In [ ]:
for engine in ["kokoro", "piper"]:
    tel_dir = f"samples/telephonie/{engine}"
    os.makedirs(tel_dir, exist_ok=True)
    for wav_file in sorted(Path(f"samples/{engine}").glob("*.wav")):
        output_file = f"{tel_dir}/{wav_file.name}"
        subprocess.run(
            ["ffmpeg", "-i", str(wav_file),
             "-ar", "8000", "-ac", "1", "-acodec", "pcm_mulaw",
             output_file, "-y"],
            check=True, capture_output=True,
        )
    print(f"{engine}: {len(list(Path(tel_dir).glob('*.wav')))} fichiers convertis en qualit\u00e9 t\u00e9l\u00e9phone")

## 6. \u00c9coute comparative

\u00c9coutez chaque phrase dans les 4 versions :
1. Kokoro qualit\u00e9 originale
2. Kokoro qualit\u00e9 t\u00e9l\u00e9phone
3. Piper qualit\u00e9 originale
4. Piper qualit\u00e9 t\u00e9l\u00e9phone

In [ ]:
from IPython.display import display, Audio, HTML

for id_phrase, texte in phrases_bot:
    display(HTML(f"<h3>{id_phrase}</h3>"))
    display(HTML(f"<p><em>{texte}</em></p>"))

    display(HTML("<b>Kokoro (originale 24kHz)</b>"))
    display(Audio(f"samples/kokoro/{id_phrase}.wav"))

    display(HTML("<b>Kokoro (t\u00e9l\u00e9phone 8kHz)</b>"))
    display(Audio(f"samples/telephonie/kokoro/{id_phrase}.wav"))

    display(HTML("<b>Piper (originale 22kHz)</b>"))
    display(Audio(f"samples/piper/{id_phrase}.wav"))

    display(HTML("<b>Piper (t\u00e9l\u00e9phone 8kHz)</b>"))
    display(Audio(f"samples/telephonie/piper/{id_phrase}.wav"))

    display(HTML("<hr>"))

## 7. Comparaison latences

In [ ]:
display(HTML("<h3>Latences de g\u00e9n\u00e9ration (ms)</h3>"))

header = f"{'Phrase':<30} {'Kokoro':>10} {'Piper':>10}"
print(header)
print("-" * len(header))
for id_phrase, _ in phrases_bot:
    k = kokoro_times.get(id_phrase, 0)
    p = piper_times.get(id_phrase, 0)
    winner = "\u2190" if k < p else "\u2192"
    print(f"{id_phrase:<30} {k:>8.0f}ms {p:>8.0f}ms  {winner}")

print("-" * len(header))
k_avg = sum(kokoro_times.values()) / len(kokoro_times)
p_avg = sum(piper_times.values()) / len(piper_times)
print(f"{'MOYENNE':<30} {k_avg:>8.0f}ms {p_avg:>8.0f}ms")

## 8. Grille d'\u00e9valuation

Remplissez cette grille apr\u00e8s \u00e9coute :

| Crit\u00e8re | Kokoro | Piper | Notes |
|---------|--------|-------|-------|
| Naturel de la voix | /5 | /5 | |
| Prosodie / rythme | /5 | /5 | |
| Clart\u00e9 en qualit\u00e9 t\u00e9l\u00e9phone | /5 | /5 | **Le plus important** |
| Prononciation \"Ma-if\" | /5 | /5 | |
| Latence | ms | ms | |
| **Score total** | /25 | /25 | |

## 9. T\u00e9l\u00e9charger les WAV

In [ ]:
# Cr\u00e9er une archive ZIP avec tous les \u00e9chantillons
!zip -r echantillons_tts.zip samples/

from google.colab import files
files.download("echantillons_tts.zip")
print("T\u00e9l\u00e9chargement lanc\u00e9.")